# ELT-08 · Validação ELT × ETL

- **Etapa:** ELT-08, Validação ELT × ETL
- **Responsável:** Pedro Wellington
- **Issue:** ELT-08
- **Entrada:** as 9 tabelas da ETL (schema `public`) e as da ELT (schema `dw_elt`)
- **Saída:** `sql/validacao/elt_checks.sql`

Compara o resultado da ELT com o da ETL, tabela por tabela: contagens, óbitos, chaves sem correspondência e diferenças de conteúdo.

## 1. Setup

In [22]:
import os
from pathlib import Path
import pandas as pd

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
os.chdir(RAIZ)   # roda a partir da raiz, para achar a pasta src/ e o arquivo .env

from src.conexao import obter_engine, testar_conexao

engine = obter_engine()
SAIDA_SQL = RAIZ / "sql" / "validacao" / "elt_checks.sql"
SAIDA_SQL.parent.mkdir(parents=True, exist_ok=True)

ETL = "public"    # schema onde a ETL carregou as tabelas
ELT = "dw_elt"    # schema onde a ELT carregou as tabelas

TABELAS = ["dim_tempo", "dim_gravidade", "dim_perfil_vitima", "dim_transporte",
           "dim_localidade", "dim_via", "dim_profissao", "dim_local_obito",
           "fato_pessoa_sinistro"]

# chave da fato -> dimensão e coluna de chave da dimensão
FKS = [("sk_data_sinistro", "dim_tempo", "sk_tempo"),
       ("sk_data_obito", "dim_tempo", "sk_tempo"),
       ("sk_gravidade", "dim_gravidade", "sk_gravidade"),
       ("sk_perfil_vitima", "dim_perfil_vitima", "sk_perfil_vitima"),
       ("sk_transporte", "dim_transporte", "sk_transporte"),
       ("sk_localidade", "dim_localidade", "sk_localidade"),
       ("sk_via", "dim_via", "sk_via"),
       ("sk_profissao", "dim_profissao", "sk_profissao"),
       ("sk_local_obito", "dim_local_obito", "sk_local_obito")]

CONSULTAS = {}   # nome -> texto SQL, para gravar no arquivo no final

def consultar(nome: str, sql: str) -> pd.DataFrame:
    """Roda a consulta no banco e guarda o texto para o arquivo elt_checks.sql."""
    CONSULTAS[nome] = sql.strip()
    return pd.read_sql(sql, engine)

_ = testar_conexao()

 Conexão estabelecida com sucesso!
 Banco de Dados : postgres
 Usuário         : postgres
 Versão SGDB     : PostgreSQL 17.11 on aarch64-unknown-linux-gnu...


In [23]:
pd.read_sql("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'dw_elt'
ORDER BY table_name
""", engine)

,table_name
0,dim_gravidade
1,dim_local_obito
2,dim_localidade
3,dim_perfil_vitima
4,dim_profissao
5,dim_tempo
6,dim_transporte
7,dim_via
8,fato_pessoa_sinistro


## 3. Contagens por tabela

In [24]:
partes = [f"SELECT '{t}' AS tabela, (SELECT COUNT(*) FROM {ETL}.{t}) AS linhas_etl, "
          f"(SELECT COUNT(*) FROM {ELT}.{t}) AS linhas_elt" for t in TABELAS]
cont = consultar('contagem_etl_elt', "\nUNION ALL ".join(partes))
cont['diferenca'] = cont['linhas_elt'] - cont['linhas_etl']
cont

,tabela,linhas_etl,linhas_elt,diferenca
0,dim_tempo,1121,1121,0
1,dim_gravidade,6,6,0
2,dim_perfil_vitima,1500,1500,0
3,dim_transporte,25,25,0
4,dim_localidade,647,647,0
5,dim_via,10,10,0
6,dim_profissao,561,561,0
7,dim_local_obito,5,5,0
8,fato_pessoa_sinistro,806977,806977,0


## 4. Total de óbitos

In [25]:
consultar('obitos_etl_elt', f"""
SELECT 'ETL' AS origem, SUM(flag_obito) AS obitos, COUNT(*) AS vitimas FROM {ETL}.fato_pessoa_sinistro
UNION ALL
SELECT 'ELT', SUM(flag_obito), COUNT(*) FROM {ELT}.fato_pessoa_sinistro
""")

,origem,obitos,vitimas
0,ETL,15793,806977
1,ELT,15793,806977


## 5. Chaves sem correspondência na ELT

In [26]:
partes = [f"SELECT '{fk} -> {dim}' AS chave, COUNT(*) AS orfaos "
          f"FROM {ELT}.fato_pessoa_sinistro f LEFT JOIN {ELT}.{dim} d ON f.{fk} = d.{pk} "
          f"WHERE d.{pk} IS NULL" for fk, dim, pk in FKS]
consultar('fk_sem_correspondencia_elt', "\nUNION ALL ".join(partes))

,chave,orfaos
0,sk_data_sinistro -> dim_tempo,0
1,sk_data_obito -> dim_tempo,0
2,sk_gravidade -> dim_gravidade,0
3,sk_perfil_vitima -> dim_perfil_vitima,0
4,sk_transporte -> dim_transporte,0
5,sk_localidade -> dim_localidade,0
6,sk_via -> dim_via,0
7,sk_profissao -> dim_profissao,0
8,sk_local_obito -> dim_local_obito,0


## 6. Diferenças de conteúdo (EXCEPT)

In [27]:
def colunas_comparaveis(tabela):
    """Colunas que existem nos dois schemas, sem as chaves sk_."""
    def cols(schema):
        df = pd.read_sql(f"""
            SELECT column_name FROM information_schema.columns
            WHERE table_schema = '{schema}' AND table_name = '{tabela}'
            ORDER BY ordinal_position""", engine)
        return list(df["column_name"])
    etl, elt = cols(ETL), cols(ELT)
    comuns = [c for c in etl if c in elt and not c.startswith("sk_")]
    return comuns, [c for c in etl if c not in elt], [c for c in elt if c not in etl]

linhas, so_um_lado = [], []
for t in TABELAS:
    comuns, so_etl, so_elt = colunas_comparaveis(t)
    if so_etl or so_elt:
        so_um_lado.append({"tabela": t, "so_na_etl": ", ".join(so_etl), "so_na_elt": ", ".join(so_elt)})
    cols = ", ".join(f'"{c}"::text AS "{c}"' for c in comuns)
    sql = f"""
SELECT '{t}' AS tabela,
  (SELECT COUNT(*) FROM (SELECT {cols} FROM {ETL}.{t} EXCEPT SELECT {cols} FROM {ELT}.{t}) a) AS so_na_etl,
  (SELECT COUNT(*) FROM (SELECT {cols} FROM {ELT}.{t} EXCEPT SELECT {cols} FROM {ETL}.{t}) b) AS so_na_elt
"""
    linhas.append(consultar(f"except_{t}", sql))

display(pd.concat(linhas, ignore_index=True))
print("Colunas que só existem em um dos lados:")
display(pd.DataFrame(so_um_lado) if so_um_lado else "nenhuma, as colunas de conteúdo são as mesmas")

,tabela,so_na_etl,so_na_elt
0,dim_tempo,0,0
1,dim_gravidade,0,0
2,dim_perfil_vitima,0,0
3,dim_transporte,0,0
4,dim_localidade,0,0
5,dim_via,0,0
6,dim_profissao,0,0
7,dim_local_obito,0,0
8,fato_pessoa_sinistro,0,0


Colunas que só existem em um dos lados:


'nenhuma, as colunas de conteúdo são as mesmas'

## 7. Conferência da fato com as dimensões

In [28]:
comuns_fato = colunas_comparaveis("fato_pessoa_sinistro")[0]
dims_cols = {dim: colunas_comparaveis(dim)[0] for dim in set(d for _, d, _ in FKS)}

def soma_verificacao(schema):
    """Um texto por pessoa (fato + conteúdo das 8 dimensões), resumido em uma soma de hashes."""
    campos = [f"""COALESCE(f."{c}"::text, '~')""" for c in comuns_fato]
    joins = []
    for fk, dim, pk in FKS:
        a = fk.replace("sk_", "d_")
        campos += [f"""COALESCE({a}."{c}"::text, '~')""" for c in dims_cols[dim]]
        joins.append(f"LEFT JOIN {schema}.{dim} {a} ON f.{fk} = {a}.{pk}")
    return (f"SELECT '{schema}' AS esquema, COUNT(*) AS linhas, "
            f"SUM(hashtextextended(concat_ws('|', {', '.join(campos)}), 0)::numeric)::text AS soma_hash "
            f"FROM {schema}.fato_pessoa_sinistro f " + " ".join(joins))

res = consultar('soma_verificacao_fato', soma_verificacao(ETL) + "\nUNION ALL\n" + soma_verificacao(ELT))
display(res)
print("Iguais:", res["linhas"].nunique() == 1 and res["soma_hash"].nunique() == 1)

,esquema,linhas,soma_hash
0,public,806977,-4020202964299140559919
1,dw_elt,806977,-4020202964299140559919


Iguais: True


## 8. Gravar o arquivo SQL

In [29]:
cabecalho = (
    "-- Consultas de validação ELT x ETL (ELT-08)\n"
    "-- Geradas pelo notebook notebooks/elt/08_validacao.ipynb\n\n"
)
blocos = [f"-- {nome}\n{sql};\n" for nome, sql in CONSULTAS.items()]
SAIDA_SQL.write_text(cabecalho + "\n".join(blocos), encoding="utf-8")
print(f"{len(CONSULTAS)} consultas gravadas em {SAIDA_SQL.relative_to(RAIZ)}")

13 consultas gravadas em sql\validacao\elt_checks.sql


## 9. Resumo

- As 9 tabelas da ELT têm o mesmo número de linhas que as da ETL (806.977 na fato).
- O total de óbitos é igual nos dois lados: 15.793.
- Nenhuma chave da fato ficou sem correspondência nas dimensões da ELT.
- O EXCEPT nas duas direções não encontrou diferença em nenhuma tabela, e as colunas comparadas são as mesmas.
- A fato junto com as dimensões também fecha: mesmo número de linhas e mesma soma de verificação nos dois schemas. Fiz essa conferência por soma porque o EXCEPT desse cruzamento estourou o disco do Supabase.
- Não há diferença para explicar. As consultas estão em `sql/validacao/elt_checks.sql`. 